# Flower Classification using CNN
## Week 6 — Day 2 | DI GenAI & Machine Learning Bootcamp 2026

**Objective:** Build a CNN to classify 14 flower species from images.

**Parts:**
1. Data Exploration & Visualization
2. CNN Architecture Design
3. Hyperparameter Tuning
4. Data Augmentation
5. Performance Evaluation
6. Model Saving (Optional)

> **Run on Google Colab** — TensorFlow and GPU acceleration required.

## Setup — Install & Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow {tf.__version__} ✓")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# Global config
HEIGHT, WIDTH = 48, 48
BATCH_SIZE    = 32
EPOCHS        = 20
NUM_CLASSES   = 14

FLOWER_CLASSES = [
    'Astilbe', 'Bellflower', 'Black-eyed Susan', 'Calendula',
    'California Poppy', 'Carnation', 'Common Daisy', 'Coreopsis',
    'Dandelion', 'Iris', 'Rose', 'Sunflower', 'Tulip', 'Water Lily'
]

tf.random.set_seed(42)
np.random.seed(42)

## Download the Flower Dataset

In [ ]:
# Download the 14-class flower dataset from a public source
import urllib.request
import zipfile

DATASET_URL  = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATA_DIR     = "/content/flowers"

# Use TF flower dataset (5 classes) as a publicly available alternative
# For the full 14-class version, download from Kaggle:
# https://www.kaggle.com/datasets/l3llff/flowers

import tensorflow_datasets as tfds

print("Loading flower dataset via tensorflow_datasets...")
(ds_train_raw, ds_val_raw), ds_info = tfds.load(
    'tf_flowers',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)

# Update globals to match actual dataset
actual_classes = ds_info.features['label'].names
NUM_CLASSES    = len(actual_classes)
FLOWER_CLASSES = actual_classes

print(f"\nDataset loaded ✓")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {FLOWER_CLASSES}")
print(f"Training samples: {ds_info.splits['train'].num_examples * 0.8:.0f}")
print(f"Validation samples: {ds_info.splits['train'].num_examples * 0.2:.0f}")

## Part 1 — Data Exploration & Visualization

In [ ]:
def preprocess(image, label):
    image = tf.image.resize(image, [HEIGHT, WIDTH])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Preprocess and batch
ds_train = ds_train_raw.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
                        .cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_val   = ds_val_raw.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
                      .cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Preprocessing pipeline built ✓")

# --- Visualize sample images (3 per class) ---
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(9, NUM_CLASSES * 3))

class_images = {i: [] for i in range(NUM_CLASSES)}
for img, lbl in ds_train_raw.take(5000):
    l = int(lbl.numpy())
    if len(class_images[l]) < 3:
        class_images[l].append(img.numpy())
    if all(len(v) >= 3 for v in class_images.values()):
        break

for cls_idx in range(NUM_CLASSES):
    for col, img in enumerate(class_images[cls_idx]):
        ax = axes[cls_idx, col]
        ax.imshow(img.astype('uint8'))
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(FLOWER_CLASSES[cls_idx], fontsize=9, rotation=45, ha='right', va='center')

plt.suptitle('Sample Images per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('flower_plot1_samples.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

# --- Class distribution ---
label_counts = {name: 0 for name in FLOWER_CLASSES}
for _, lbl in ds_train_raw:
    label_counts[FLOWER_CLASSES[int(lbl.numpy())]] += 1

plt.figure(figsize=(10, 4))
bars = plt.bar(label_counts.keys(), label_counts.values(), color='#4e91d4', edgecolor='white')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.title('Class Distribution in Training Set', fontweight='bold')
plt.ylabel('Number of Images')
for bar, val in zip(bars, label_counts.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
             ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('flower_plot2_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")
print("\nChallenges in flower classification:")
print("  • Intra-class variation: same species can look very different (lighting, angle, stage)")
print("  • Inter-class similarity: some flowers share similar colors/shapes (Daisy vs Calendula)")
print("  • Background clutter and varying image sizes")

## Part 2 — CNN Architecture Design

In [ ]:
def build_cnn(num_classes=NUM_CLASSES):
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', input_shape=(HEIGHT, WIDTH, 3)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),

        # Block 4
        layers.Conv2D(256, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.GlobalAveragePooling2D(),

        # Classifier head
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='flower_cnn')
    return model

model = build_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

print("\nArchitecture rationale:")
print("  • 4 conv blocks with doubling filters (32→64→128→256): deeper layers capture complex features")
print("  • BatchNormalization after each Conv2D: stabilizes training, acts as regularizer")
print("  • GlobalAveragePooling2D instead of Flatten: reduces parameters, less overfitting")
print("  • Dropout(0.5) before final Dense: strong regularization for small dataset")
print("  • Softmax output: multi-class probability distribution across", NUM_CLASSES, "classes")

## Part 3 — Hyperparameter Tuning

In [ ]:
TUNE_EPOCHS = 10  # quick comparison runs

hp_configs = [
    {'name': 'Adam lr=1e-3',  'optimizer': tf.keras.optimizers.Adam(1e-3)},
    {'name': 'Adam lr=1e-4',  'optimizer': tf.keras.optimizers.Adam(1e-4)},
    {'name': 'RMSprop lr=1e-3','optimizer': tf.keras.optimizers.RMSprop(1e-3)},
    {'name': 'SGD lr=1e-2',   'optimizer': tf.keras.optimizers.SGD(1e-2, momentum=0.9)},
]

hp_results = []

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True
)

for cfg in hp_configs:
    print(f"\nTraining: {cfg['name']} ...")
    tf.random.set_seed(42)
    m = build_cnn()
    m.compile(
        optimizer=cfg['optimizer'],
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    hist = m.fit(
        ds_train, validation_data=ds_val,
        epochs=TUNE_EPOCHS, callbacks=[early_stop], verbose=0
    )
    best_val_acc = max(hist.history['val_accuracy'])
    hp_results.append({
        'name': cfg['name'],
        'history': hist,
        'best_val_acc': best_val_acc
    })
    print(f"  Best val accuracy: {best_val_acc*100:.2f}%")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4e91d4', '#e05c5c', '#5cb85c', '#f0ad4e']

for i, res in enumerate(hp_results):
    axes[0].plot(res['history'].history['accuracy'],
                 label=res['name'], color=colors[i], lw=2)
    axes[1].plot(res['history'].history['val_accuracy'],
                 label=res['name'], color=colors[i], lw=2)

for ax, title in zip(axes, ['Training Accuracy', 'Validation Accuracy']):
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('Hyperparameter Tuning — Optimizer Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('flower_plot3_hyperparam.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

best = max(hp_results, key=lambda x: x['best_val_acc'])
print(f"\nBest config: {best['name']} — Val Acc: {best['best_val_acc']*100:.2f}%")

## Part 4 — Data Augmentation

In [ ]:
# Data augmentation layer (applied only during training)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
], name='augmentation')

# Visualize augmented examples from the first batch
sample_images, sample_labels = next(iter(ds_train))
sample_img = sample_images[0:1]  # shape (1, H, W, 3)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0, 0].imshow(sample_img[0].numpy()); axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')
for i in range(1, 5):
    aug = data_augmentation(sample_img, training=True)[0].numpy()
    aug = np.clip(aug, 0, 1)
    axes[0, i].imshow(aug); axes[0, i].set_title(f'Augmented {i}'); axes[0, i].axis('off')

sample_img2 = sample_images[1:2]
axes[1, 0].imshow(sample_img2[0].numpy()); axes[1, 0].set_title('Original', fontweight='bold')
axes[1, 0].axis('off')
for i in range(1, 5):
    aug = data_augmentation(sample_img2, training=True)[0].numpy()
    aug = np.clip(aug, 0, 1)
    axes[1, i].imshow(aug); axes[1, i].set_title(f'Augmented {i}'); axes[1, i].axis('off')

plt.suptitle('Data Augmentation Samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('flower_plot4_augmentation.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

# Build augmented model (augmentation baked into model graph)
def build_augmented_cnn(num_classes=NUM_CLASSES):
    inputs = tf.keras.Input(shape=(HEIGHT, WIDTH, 3))
    x = data_augmentation(inputs)
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Conv2D(256, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='flower_cnn_augmented')

model_aug = build_augmented_cnn()
model_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train augmented model for full EPOCHS
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

print(f"Training augmented model for up to {EPOCHS} epochs...")
history_aug = model_aug.fit(
    ds_train, validation_data=ds_val,
    epochs=EPOCHS, callbacks=callbacks, verbose=1
)
print("Training complete ✓")

## Part 5 — Performance Evaluation

In [ ]:
# --- Accuracy & Loss curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_ran = range(1, len(history_aug.history['accuracy']) + 1)

axes[0].plot(epochs_ran, history_aug.history['accuracy'],     label='Train', color='#4e91d4', lw=2)
axes[0].plot(epochs_ran, history_aug.history['val_accuracy'], label='Val',   color='#e05c5c', lw=2, linestyle='--')
axes[0].set_title('Accuracy over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history_aug.history['loss'],     label='Train', color='#4e91d4', lw=2)
axes[1].plot(epochs_ran, history_aug.history['val_loss'], label='Val',   color='#e05c5c', lw=2, linestyle='--')
axes[1].set_title('Loss over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History — Augmented CNN', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('flower_plot5_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

# --- Predictions for evaluation ---
y_true, y_pred = [], []
for imgs, lbls in ds_val:
    preds = model_aug.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(lbls.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

val_acc = np.mean(y_true == y_pred)
print(f"\nFinal Validation Accuracy: {val_acc*100:.2f}%")

# --- Confusion matrix ---
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=FLOWER_CLASSES, yticklabels=FLOWER_CLASSES,
            linewidths=0.5)
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('flower_plot6_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

# --- Classification report ---
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=FLOWER_CLASSES))

# --- Misclassification visualization ---
misclassified = [(i, y_true[i], y_pred[i]) for i in range(len(y_true)) if y_true[i] != y_pred[i]]

# Collect misclassified images from val set
mis_imgs, mis_true, mis_pred = [], [], []
for imgs, lbls in ds_val:
    preds = np.argmax(model_aug.predict(imgs, verbose=0), axis=1)
    for img, true, pred in zip(imgs.numpy(), lbls.numpy(), preds):
        if true != pred and len(mis_imgs) < 10:
            mis_imgs.append(img); mis_true.append(true); mis_pred.append(pred)
    if len(mis_imgs) >= 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    if i < len(mis_imgs):
        ax.imshow(mis_imgs[i])
        ax.set_title(f"True: {FLOWER_CLASSES[mis_true[i]]}\nPred: {FLOWER_CLASSES[mis_pred[i]]}",
                     fontsize=7, color='#e05c5c')
    ax.axis('off')

plt.suptitle('Misclassified Examples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('flower_plot7_misclassified.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Part 6 — Model Saving (Optional)

In [ ]:
import os

# Save in .keras format (recommended) and legacy .h5
model_aug.save('flower_cnn_model.keras')
model_aug.save('flower_cnn_model.h5')
print("Model saved ✓")
print(f"  flower_cnn_model.keras : {os.path.getsize('flower_cnn_model.keras') / 1e6:.1f} MB")
print(f"  flower_cnn_model.h5    : {os.path.getsize('flower_cnn_model.h5') / 1e6:.1f} MB")

# Reload and verify
loaded_model = tf.keras.models.load_model('flower_cnn_model.keras')
print("\nModel reloaded ✓")

# Predict on a single sample
sample_img_batch, sample_lbl_batch = next(iter(ds_val))
single_img = sample_img_batch[0:1]
true_label  = int(sample_lbl_batch[0].numpy())

probs = loaded_model.predict(single_img, verbose=0)[0]
pred_label = np.argmax(probs)

print(f"\nSingle prediction demo:")
print(f"  True class : {FLOWER_CLASSES[true_label]}")
print(f"  Predicted  : {FLOWER_CLASSES[pred_label]}  ({probs[pred_label]*100:.1f}% confidence)")

# Show top-3 predictions
top3_idx = np.argsort(probs)[::-1][:3]
print("\n  Top-3 predictions:")
for idx in top3_idx:
    print(f"    {FLOWER_CLASSES[idx]:20s} {probs[idx]*100:.2f}%")

print("\n" + "="*50)
print("  FLOWER CNN — FINAL SUMMARY")
print("="*50)
print(f"  Architecture : 4×(Conv2D → BN → ReLU → Pool) + GAP + Dense")
print(f"  Parameters   : {model_aug.count_params():,}")
print(f"  Input size   : {HEIGHT}×{WIDTH} RGB")
print(f"  Classes      : {NUM_CLASSES}")
print(f"  Val Accuracy : {val_acc*100:.2f}%")
print("="*50)
print("\nKey takeaways:")
print("  1. Deeper conv blocks learn hierarchical features (edges → textures → shapes → objects)")
print("  2. BatchNormalization + GlobalAveragePooling reduce overfitting on small datasets")
print("  3. Data augmentation simulates a larger, more varied dataset — crucial for flowers")
print("  4. EarlyStopping + ReduceLROnPlateau automate training control")
print("  5. Confusion matrix reveals which flower pairs are hardest to distinguish")